# Visualisation of Machine Learning results (Classification)

<a id='Index'></a>
# Exercise list:
* <a href='#Exercise-01'>Exercise-01</a>
* <a href='#Exercise-02'>Exercise-02</a>
* <a href='#Exercise-03'>Exercise-03</a>
* <a href='#Exercise-04'>Exercise-04</a>

In [1]:
%spark 16 32g

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/01 17:08:01 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.4.2
      /_/

Using Python version 3.9.18 (main, Dec 15 2023 17:48:57)
Spark context Web UI available at None
Spark context available as 'sc' (master = local[*], app id = local-1759331281938).
SparkSession available as 'spark'.


In [2]:
%run handlers.ipynb

Pandas version:  2.1.4


Additionally for this visualisation notebook:
* Import **geopy** to get the coordinates of the stations.
* Import **folium** to visualise the data on a map.

In [3]:
import colorsys
from collections import Counter
from IPython.display import HTML
from geopy.geocoders import Nominatim
import folium

# Prepare the data

* Read in **df_test_accu_ds ... .csv**. 
  * This DataFrame contains the classification results (**accuracy** of the prediction) over the test dataset.
  * All data of the S1 line are considered, clustered by station.
  * The model has been trained  with **5 different sets of features**.
  * Pre-saved data are in the read-only directory **local_data_dir_ro**. Data you produced are in **local_data_dir**.

In [4]:
path = os.path.join(LustrePrefix, local_data_dir_ro,'df_test_accu_ds'+suffix_s_w+'.csv')
df_test_accu = spark.read.option("header", True).option('inferSchema',True).csv(path, header = True)

<a href='#Index'>Exercise Index</a>
<a id='Exercise-01'></a>

**EXERCISE 1**

* Transform this Spark DataFrame into a **pandas** DataFrame with toPandas()
* Display the whole DataFrame with iloc[\__,__]

In [5]:
df_accu_test_pd = df_test_accu.toPandas()
df_accu_test_pd.iloc[:, :]

,ds,accu1,accu2,accu3,accu4,accu5
0,TWER,0.690909,0.685195,0.587532,0.587532,0.587532
1,TSRO,0.611277,0.579667,0.888936,0.888936,0.888936
2,TWD,0.405752,0.403184,0.881870,0.881870,0.881870
3,TSMI,0.618854,0.623290,0.857301,0.857301,0.857301
4,TP,0.324710,0.319430,0.472545,0.472545,0.472545
5,THUB,0.721491,0.705044,0.932018,0.932018,0.932018
6,TGOL,0.737263,0.741053,0.781474,0.781474,0.781895
7,TSU,0.616654,0.656640,0.895084,0.895084,0.894718
8,TS T,0.648547,0.672329,0.815780,0.815780,0.815780
9,TKTO,0.163184,0.173134,0.243781,0.243781,0.258706


# Get the coordinates of the stations
* In order to draw the S-Bahn stations on a map, we need to know their geographical coordinates.
* To locate and label the stations on the map, we need their **complete names**:
  * Read-in the database of stations as a pandas DataFrame
  * Convert it into a **python dictionary**.
* (Already done in Notebook1, EX1)

In [6]:
filename_stations_csv = 'StationData.csv'
path_stations_csv = os.path.join(basedir, filename_stations_csv)
stations_csv_pd = pd.read_csv(os.path.join(LustrePrefix,path_stations_csv), sep = ';', encoding = "ISO-8859-1")
abbr_stations = stations_csv_pd.set_index('DS100')['NAME'].to_dict()

print('Some dictionary key (DS100) : value (full name):')
for i in range(5):
    print(list(abbr_stations.items())[i][0], ': ', list(abbr_stations.items())[i][1])
print('...')

Some dictionary key (DS100) : value (full name):
TA :  Aalen Hbf
TAU :  Aulendorf
TB :  Backnang
TBF :  Bad Friedrichshall Hbf
TBM :  Bietigheim-Bissingen
...


* **Without direct Internet access from the cluster**, we need to read in the coordinates from the Excel table S-Bahn-coordinates.xlsx 

In [7]:
filename_stations = 'S-Bahn-coordinates.xlsx'
path_stations = os.path.join(basedir, filename_stations)

df_pd_stations_excel = pd.ExcelFile(os.path.join(LustrePrefix,path_stations))
df_pd_stations = pd.read_excel(df_pd_stations_excel)

# Print to screen 5 rows of this Pandas DataFrame
df_pd_stations.iloc[:5,:]

,Station,Abk.,Eröffnung,Beginn des,Linien,Art,Gl.,Bahnsteighöhe [cm],Umstieg,Stadt/Gemeinde,Lk.,Latitude,Longitude
0,Altbach,TACH,14. Dez. 1846,1. Okt. 1978,S 1,Hp,2,76 cm,NaN,Altbach,ES,48.720976,9.382063
1,Asperg,TAX,11. Okt. 1847,31. Mai 1981,S 5,Bf,2,96 cm,NaN,Asperg,LB,48.906599,9.147447
2,Backnang,TB,26. Okt. 1876,27. Sep. 1981,S 3 S 4,Bf,2,76 cm(5)/96 cm(1),RV,Backnang,WN,48.942613,9.426545
3,Bad Cannstatt,TSC,22. Okt. 1845,1. Okt. 1978,S 1 S 2 S 3 S 11,Bf,2,96 cm,"RV, U",Stuttgart,S,48.801313,9.217677
4,Benningen (Neckar),TBEN,8. Dez. 1879,28. Sep. 1980,S 4,Bf,2,96 cm,NaN,Benningen am Neckar,LB,48.943733,9.244395


# Ordered sequence of S1 stations
* We need an ORDERED, COMPLETE sequence of stations of the line S1 as a python LIST (**list_S1_ds_seq**).
* We provide the ordered list of DS100 codes.

In [8]:
list_S1_ds_seq = ['THE', 'TNUF', 'TGT', 'TEHN', 'THUB', 'TBO', 'TGOL', \
                 'TSRO', 'TSV', 'TSOS', 'TSUN', 'TSS', 'TSFS', 'TSMI', \
                 'TS  T', 'TS', 'TSC', 'TSNS', 'TSU', 'TSOM', 'TEME', 'TE', \
                 'TOES', 'TEZL', 'TACH', 'TP', 'TWER', 'TWD', 'TKTO', 'TKT']
print("All", len(list_S1_ds_seq), "stations served by the S1 (ordered):\n", list_S1_ds_seq)

All 30 stations served by the S1 (ordered):
 ['THE', 'TNUF', 'TGT', 'TEHN', 'THUB', 'TBO', 'TGOL', 'TSRO', 'TSV', 'TSOS', 'TSUN', 'TSS', 'TSFS', 'TSMI', 'TS  T', 'TS', 'TSC', 'TSNS', 'TSU', 'TSOM', 'TEME', 'TE', 'TOES', 'TEZL', 'TACH', 'TP', 'TWER', 'TWD', 'TKTO', 'TKT']


<a href='#Index'>Exercise Index</a>
<a id='Exercise-02'></a>

**EXERCISE 2**

* We also define **list_S1_ds** as a list of the DS100 codes of the S1 stations,
* ... but corresponding to the sequence in the accuracy DataFrame of the test data.
* Use tolist() to convert the 'ds' column into a python list.

In [9]:
list_S1_ds = df_accu_test_pd['ds'].tolist()
print(len(list_S1_ds), "stations served by the S1 according to df_accu_test_pd:\n", list_S1_ds)

27 stations served by the S1 according to df_accu_test_pd:
 ['TWER', 'TSRO', 'TWD', 'TSMI', 'TP', 'THUB', 'TGOL', 'TSU', 'TS  T', 'TKTO', 'TGT', 'TSV', 'TOES', 'TE', 'TSUN', 'TSFS', 'TEHN', 'TSOS', 'TACH', 'TSC', 'TEZL', 'TSNS', 'TEME', 'TBO', 'TSS', 'TSOM', 'TNUF']


# Plot results based on all S1 data
Visualise **on a map** the accuracy of the delay prediction at each S1 station.

Set a **threshold** for the map colour code:

* accuracy <  threshold[1] : prediction is "black"
* threshold[1] <= accuracy < threshold[0] : prediction is "orange"
* accuracy >= threshold[0] :                prediction is "green"

In [10]:
threshold = [0.8, 0.5]

Define the filenames for the plots:

In [11]:
filename = ['AvgAccuracy_1_S1' + suffix_s_w +'.html',\
            'AvgAccuracy_2_S1' + suffix_s_w +'.html',\
            'AvgAccuracy_3_S1' + suffix_s_w +'.html',\
            'AvgAccuracy_4_S1' + suffix_s_w +'.html',\
            'AvgAccuracy_5_S1' + suffix_s_w +'.html']
accu_nums=['accu1','accu2','accu3','accu4','accu5']

# Option 1: Does not require web access from cluster.

<a href='#Index'>Exercise Index</a>
<a id='Exercise-03'></a>

**EXERCISE 3 (to be done in HANDLERS)**

Two functions are needed (in handlers):
1. **getCoordsNoInt**: 
   * It takes the **coordinates** of the stations as input and **associates a color** according to the delay threshold.
   * getCoordsNoInt takes the following as input:
     - DS100 codes of all ordered stations (list_S1_ds_seq) 
     - Pandas DataFrame with station information
     - list of accuracies at each station for the model considered
     - dictionary of station names
     - DS100 codes of (unordered) stations associated to the accuracy DataFrame (list_S1_ds)
     - threshold


2. **drawMapDic**, which draws the map with folium.

* Do the exercises in **handlers** (function **getCoordsNoInt**)!
* Then re-run handlers and do the plot (i.e., execute the cells below).

* First of all, produce and check the plot corresponding to the **first feature combination**.
* Note that these plots are saved as **html files** and have to be **downloaded locally** to be visualised.

In [13]:
%run handlers.ipynb

for i in range(1):
    path = os.path.join(local_classification_plot_dir, filename[i])
    coords, colors = PredictionVisualize.getCoordsNoInt(list_S1_ds_seq, df_pd_stations, df_accu_test_pd[accu_nums[i]].tolist(),\
                                                   abbr_stations, list_S1_ds, threshold)
    print('Found coordinates and associated colors for set', i+1, ' of ', len(filename))
    PredictionVisualize.drawMapDic(coords, colors, list_S1_ds_seq,list_S1_ds, abbr_stations).save(path)
    print('Done drawing map', i+1, ' of ', len(filename))

Pandas version:  2.1.4
Nufringen
[48.620241, 8.8896066]
Gärtringen
[48.6411071, 8.9090142]
Ehningen(b Böblingen)
[48.6619947, 8.9433915]
Hulb
[48.6792228, 8.982399]
Böblingen
[48.688044, 9.0047319]
Goldberg(Württ)
[48.6958639, 9.0187569]
Stuttgart-Rohr
[48.7178289, 9.1084095]
Stuttgart-Vaihingen
[48.7264626, 9.1131764]
Stuttgart-Österfeld
[48.7384764, 9.1154715]
Stuttgart Universität
[48.7453828, 9.1032675]
Stuttgart Schwabstr.
[48.7703429, 9.1564714]
Stuttgart Feuersee
[48.7725298, 9.1662246]
Stuttgart Stadtmitte
[48.7758379, 9.1723739]
Stuttgart Hbf (tief)
[48.7836248, 9.1816559]
Stuttgart-Bad Cannstatt
[48.8013135, 9.2176766]
Stuttgart Neckarpark
[48.7919493, 9.241149]
Stuttgart-Untertürkheim
[48.7795105, 9.2496215]
Stuttgart-Obertürkheim
[48.76184, 9.267911]
Esslingen-Mettingen
[48.7471967, 9.2760051]
Esslingen(Neckar)
[48.73885, 9.3006986]
Oberesslingen
[48.7300539, 9.3272098]
Esslingen-Zell
[48.7243147, 9.3594531]
Altbach
[48.7209765, 9.3820629]
Plochingen
[48.7129945530309, 9.41

* If the resulting plot of the block above is correct, produce **all 5 plots**:

In [14]:
for i in range(len(filename)):
    path = os.path.join(local_classification_plot_dir, filename[i])
    coords, colors = PredictionVisualize.getCoordsNoInt(list_S1_ds_seq, df_pd_stations, df_accu_test_pd[accu_nums[i]].tolist(),\
                                                   abbr_stations, list_S1_ds, threshold)
    print('Found coordinates and associated colors for set', i+1, ' of ', len(filename))
    PredictionVisualize.drawMapDic(coords, colors, list_S1_ds_seq,list_S1_ds, abbr_stations).save(path)
    print('Done drawing map', i+1, ' of ', len(filename))

Nufringen
[48.620241, 8.8896066]
Gärtringen
[48.6411071, 8.9090142]
Ehningen(b Böblingen)
[48.6619947, 8.9433915]
Hulb
[48.6792228, 8.982399]
Böblingen
[48.688044, 9.0047319]
Goldberg(Württ)
[48.6958639, 9.0187569]
Stuttgart-Rohr
[48.7178289, 9.1084095]
Stuttgart-Vaihingen
[48.7264626, 9.1131764]
Stuttgart-Österfeld
[48.7384764, 9.1154715]
Stuttgart Universität
[48.7453828, 9.1032675]
Stuttgart Schwabstr.
[48.7703429, 9.1564714]
Stuttgart Feuersee
[48.7725298, 9.1662246]
Stuttgart Stadtmitte
[48.7758379, 9.1723739]
Stuttgart Hbf (tief)
[48.7836248, 9.1816559]
Stuttgart-Bad Cannstatt
[48.8013135, 9.2176766]
Stuttgart Neckarpark
[48.7919493, 9.241149]
Stuttgart-Untertürkheim
[48.7795105, 9.2496215]
Stuttgart-Obertürkheim
[48.76184, 9.267911]
Esslingen-Mettingen
[48.7471967, 9.2760051]
Esslingen(Neckar)
[48.73885, 9.3006986]
Oberesslingen
[48.7300539, 9.3272098]
Esslingen-Zell
[48.7243147, 9.3594531]
Altbach
[48.7209765, 9.3820629]
Plochingen
[48.7129945530309, 9.41184664689429]
Wernau(Ne

# Option 2: Requires web access from cluster.
### Normally, not suitable for working on cluster. Locally on your workstation, it should work.

<a href='#Index'>Exercise Index</a>
<a id='Exercise-04'></a>

**EXERCISE 4 (optional, to be done in HANDLERS)**
* Do the exercises in **handlers** (function **getCoords**)!
* Then re-run handlers and do the plot (i.e., execute the cell below).

Two functions are needed:
1. **getCoords**: 
   * Computes the **coordinates** of the stations and **associates a color** according to the delay threshold.
   * getCoords takes the following as input:
     - DS100 codes of all ordered stations (list_S1_ds_seq) 
     - list of accuracies at each station for the model considered
     - dictionary of station names
     - DS100 codes of (unordered) stations associated to the accuracy DataFrame (list_S1_ds)
     - threshold

2. **drawMapDic**, which draws the map with folium.

* First of all, produce and check the plot corresponding to the **first feature combination**.
* Notice that these plots are saved as **html files** and have to be **downloaded locally** to be visualised.

In [ ]:
%run handlers.ipynb

for i in list(range(1)):
    path = os.path.join(local_classification_plot_dir, filename[i])
    coords, colors = PredictionVisualize.getCoords(list_S1_ds_seq, df_accu_test_pd[accu_nums[i]].tolist(),\
                                                   abbr_stations, list_S1_ds, threshold)
    print('Found coordinates and associated colors for set', i+1, ' of ', len(filename))
    PredictionVisualize.drawMapDic(coords, colors, list_S1_ds_seq,list_S1_ds, abbr_stations).save(path)
    print('Done drawing map', i+1, ' of ', len(filename))

* If the resulting plot of the block above is correct, produce **all 5 plots**:

In [ ]:
for i in list(range(len(filename))):
    path = os.path.join(local_classification_plot_dir, filename[i])
    coords, colors = PredictionVisualize.getCoords(list_S1_ds_seq, df_accu_test_pd[accu_nums[i]].tolist(),\
                                                   abbr_stations, list_S1_ds, threshold)
    print('Found coordinates and associated colors for set', i+1, ' of ', len(filename))
    PredictionVisualize.drawMapDic(coords, colors, list_S1_ds_seq,list_S1_ds, abbr_stations).save(path)
    print('Done drawing map', i+1, ' of ', len(filename))